This notebook contains the analysis of the United Nations Security Council ceasefire, sanctions, and humanitarian assistance resolutions datasets.

**Dataset Access**

The datasets consist of PDF transcripts of UNSC meetings, organized by resolution type:

- ceasefire

- sanctions

- humanitarian_assistance

The data is not stored directly in the drive. Instead, it is downloaded automatically from shared archives when the notebook is run. So no need to download or upload these datasets from anywhere

**How Data Is Loaded**

The notebook includes an ensure_data() function that:

Checks whether the required datasets exist in the local data/ directory.

Downloads compressed archives from a public source if the data is missing.

Extracts the PDFs into the correct folder structure:

- data/
  ├── ceasefire/
  ├── sanctions/
  └── humanitarian_assistance/


In [10]:
!pip install pdfplumber
!pip install bertopic
!pip -q install gdown

In [11]:
import pandas as pd
import os
import re
import glob
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import pdfplumber
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import zipfile
import shutil


In [12]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [13]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Import and Early Preprocessing

The data is in zipped folder which anyone can have access to. The following cell downloads the zip, unzip and access the data pdf files.

In [43]:
BASE_DATA_DIR = Path("data")

DRIVE_ZIPS = {
    "humanitarian_assistance": "1mmXGA2Ug_KxL_AuLwXFdIcuCS5NSzQIx",
    "sanctions": "1LEaxZ1gWGRnjkVSRD2mCYWm4sdZQOL_y",
    "ceasefire": "1lkth8ngBn_GoRwSQiZKiWOCXV049kqPD",
}

def download_and_extract(name: str, file_id: str):
    BASE_DATA_DIR.mkdir(exist_ok=True)
    out_dir = BASE_DATA_DIR / name
    if out_dir.exists() and any(out_dir.iterdir()):
        return  # already present

    zip_path = BASE_DATA_DIR / f"{name}.zip"
    url = f"https://drive.google.com/uc?id={file_id}"

    print(f"Downloading {name}...")
    gdown.download(url, str(zip_path), quiet=False)

    print(f"Extracting {name}...")
    out_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    zip_path.unlink()  # delete zip after extracting

def ensure_data():
    for name, file_id in DRIVE_ZIPS.items():
        download_and_extract(name, file_id)

ensure_data()

def ensure_data():
    for name, file_id in DRIVE_ZIPS.items():
        download_and_extract(name, file_id)

ensure_data()


Downloading...
From (original): https://drive.google.com/uc?id=1mmXGA2Ug_KxL_AuLwXFdIcuCS5NSzQIx
From (redirected): https://drive.google.com/uc?id=1mmXGA2Ug_KxL_AuLwXFdIcuCS5NSzQIx&confirm=t&uuid=1c5463ca-f3e8-4b30-b931-c370512e6514
To: /content/data/humanitarian_assistance.zip
100%|██████████| 151M/151M [00:02<00:00, 75.1MB/s]


Extracting humanitarian_assistance...


Downloading...
From (original): https://drive.google.com/uc?id=1LEaxZ1gWGRnjkVSRD2mCYWm4sdZQOL_y
From (redirected): https://drive.google.com/uc?id=1LEaxZ1gWGRnjkVSRD2mCYWm4sdZQOL_y&confirm=t&uuid=6e729a72-b1e4-49e3-9cb7-acfd43c69c06
To: /content/data/sanctions.zip
100%|██████████| 250M/250M [00:02<00:00, 120MB/s]


Extracting sanctions...


Downloading...
From (original): https://drive.google.com/uc?id=1lkth8ngBn_GoRwSQiZKiWOCXV049kqPD
From (redirected): https://drive.google.com/uc?id=1lkth8ngBn_GoRwSQiZKiWOCXV049kqPD&confirm=t&uuid=526a6773-4ddb-477b-ad76-0a0e8a0684bc
To: /content/data/ceasefire.zip
100%|██████████| 385M/385M [00:02<00:00, 141MB/s]


Extracting ceasefire...


In [44]:
HUMAN_ASSIST_PATH = BASE_DATA_DIR / "humanitarian_assistance"
SANCTIONS_PATH = BASE_DATA_DIR / "sanctions"
CEASEFIRE_PATH = BASE_DATA_DIR / "ceasefire"

TOPIC_FOLDERS = {
    "Ceasefire": CEASEFIRE_PATH,
    "Humanitarian Assistance": HUMAN_ASSIST_PATH,
    "Sanctions": SANCTIONS_PATH,
}

In [45]:
# check the pdf files exist in the exracted zipped folders
for topic, path in TOPIC_FOLDERS.items():
    print(f"{topic}: exists={path.exists()}, pdfs={len(list(path.glob('*.pdf')))}")


Ceasefire: exists=True, pdfs=526
Humanitarian Assistance: exists=True, pdfs=601
Sanctions: exists=True, pdfs=941


In [5]:
# HUMAN_ASSIT_PATH = "/content/drive/MyDrive/ANLP - Project/Datasets/Humanitarian Assistance"
# SANCTIONS_PATH = "/content/drive/MyDrive/ANLP - Project/Datasets/Sanctions"
# CEASEFIRE_PATH = "/content/drive/MyDrive/ANLP - Project/Datasets/Ceasefire"


# TOPIC_FOLDERS = {
#     "Ceasefire": CEASEFIRE_PATH,
#     "Humanitarian Assistance": HUMAN_ASSIT_PATH,
#     "Sanctions": SANCTIONS_PATH,
# }

In [46]:
def extract_pdf_text(pdf_path):
    """
    Extract text from a PDF file, page by page.
    Returns a list of dicts: [{'page': page_num, 'text': text}, ...]
    """
    text_by_page = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                if text:
                    text_by_page.append({
                        "page": page_num,
                        "text": text
                    })
    except Exception as e:
        print(f"Error reading {os.path.basename(pdf_path)}: {e}")
        return []

    return text_by_page


def extract_full_text(pdf_path):
    """
    Flatten the per-page text into a single big string.
    """
    pages = extract_pdf_text(pdf_path)
    if not pages:
        return ""
    return "\n".join(p["text"] for p in pages)

In [47]:
def build_meeting_index(topic_folders):
    """
    Given a dict {topic: folder_path}, build an index of meetings.

    Returns:
        meetings: dict keyed by meeting_id, e.g.
            {
              "S_PV-4515-EN": {
                  "meeting_id": ...,
                  "topic": "Humanitarian Assistance",
                  "pdf_path": "/content/.../S_PV-4515-EN.pdf"
              },
              ...
            }
    """
    meetings = {}

    for topic, folder in topic_folders.items():
        pdf_pattern = os.path.join(folder, "*.pdf")
        pdf_files = sorted(glob.glob(pdf_pattern))

        print(f"\n📂 Topic: {topic}")
        print(f"   Found {len(pdf_files)} PDFs in {folder}")

        for pdf_path in pdf_files:
            filename = os.path.basename(pdf_path)
            meeting_id = os.path.splitext(filename)[0]  # e.g. "S_PV-4515-EN"

            meetings[meeting_id] = {
                "meeting_id": meeting_id,
                "topic": topic,
                "pdf_path": pdf_path,
            }

    print(f"\n✅ Total meetings indexed: {len(meetings)}")
    return meetings


meetings_index = build_meeting_index(TOPIC_FOLDERS)



📂 Topic: Ceasefire
   Found 526 PDFs in data/ceasefire

📂 Topic: Humanitarian Assistance
   Found 601 PDFs in data/humanitarian_assistance

📂 Topic: Sanctions
   Found 941 PDFs in data/sanctions

✅ Total meetings indexed: 1861


In [48]:
def segment_speeches(transcript_text):
    """
    Segment the meeting transcript into individual speeches.
    Identifies speakers and their countries.
    """
    speeches = []

    # Pattern to identify speaker headers
    # More comprehensive patterns for various formats
    patterns = [
        r'(Mr\.|Ms\.|Mrs\.|Sir|Dr\.|Ambassador)\s+([A-Z][A-Za-z\-\s]+)\s*\(([^)]+)\)\s*:',
        r'(Mr\.|Ms\.|Mrs\.|Sir|Dr\.|Ambassador)\s+([A-Z][A-Z\-\s]+)\s*\(([^)]+)\)\s*:',
        r'The\s+PRESIDENT\s*:',  # Special case for President
    ]

    lines = transcript_text.split('\n')

    current_speaker = None
    current_country = None
    current_speech = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Check if this line starts a new speech
        speaker_found = False

        for pattern in patterns:
            speaker_match = re.search(pattern, line)

            if speaker_match:
                speaker_found = True

                # Save previous speech if exists
                if current_speaker and current_speech:
                    speeches.append({
                        'speaker': current_speaker,
                        'country': current_country,
                        'text': ' '.join(current_speech).strip()
                    })

                # Handle President case
                if 'PRESIDENT' in line:
                    current_speaker = 'PRESIDENT'
                    current_country = 'Chair'
                else:
                    current_speaker = speaker_match.group(2).strip()
                    current_country = speaker_match.group(3).strip()

                # Get text after the colon
                colon_pos = line.find(':', speaker_match.start())
                if colon_pos != -1:
                    remaining_text = line[colon_pos+1:].strip()
                    current_speech = [remaining_text] if remaining_text else []
                else:
                    current_speech = []

                break

        # If no new speaker found, continue current speech
        if not speaker_found and current_speaker:
            # Skip page numbers, headers, and very short lines
            if (len(line) > 3 and
                not re.match(r'^\d+$', line) and
                not re.match(r'^[A-Z]/[a-z]+$', line)):  # Skip lines like "JB/10"
                current_speech.append(line)

    # Add final speech
    if current_speaker and current_speech:
        speeches.append({
            'speaker': current_speaker,
            'country': current_country,
            'text': ' '.join(current_speech).strip()
        })

    return speeches

This will take about an hour to process about 1500 meeting documents

In [ ]:
def process_meetings_speeches_only(meetings_index):
    """
    For each meeting (one PDF), extract the full text and segment into speeches.
    Returns a dict:
        {
          meeting_id: {
             'meeting_id': ...,
             'topic': ...,
             'pdf_path': ...,
             'speeches': [ {speaker, country, text}, ... ]
          },
          ...
        }
    """
    processed = {}

    for meeting_id, info in meetings_index.items():
        print(f"\n📄 Processing {meeting_id} ({info['topic']})")
        transcript_text = extract_full_text(info["pdf_path"])

        if not transcript_text:
            print("   ⚠️ Could not extract any text from PDF.")
            continue

        speeches = segment_speeches(transcript_text)
        print(f"   ✓ Extracted {len(speeches)} speeches")

        processed[meeting_id] = {
            "meeting_id": meeting_id,
            "topic": info["topic"],
            "pdf_path": info["pdf_path"],
            "speeches": speeches,
        }

    print(f"\n✅ Finished. Processed {len(processed)} meetings.")
    return processed


processed_meetings = process_meetings_speeches_only(meetings_index)



📄 Processing S_PV-1166-EN (Ceasefire)
   ✓ Extracted 7 speeches

📄 Processing S_PV-1227-EN (Ceasefire)
   ✓ Extracted 10 speeches

📄 Processing S_PV-1231-EN (Ceasefire)
   ✓ Extracted 6 speeches

📄 Processing S_PV-1232-EN (Ceasefire)
   ✓ Extracted 3 speeches

📄 Processing S_PV-1233-EN (Ceasefire)
   ✓ Extracted 2 speeches

📄 Processing S_PV-1237-EN (Ceasefire)
   ✓ Extracted 27 speeches

📄 Processing S_PV-1238-EN (Ceasefire)
   ✓ Extracted 10 speeches

📄 Processing S_PV-1239-EN (Ceasefire)
   ✓ Extracted 8 speeches

📄 Processing S_PV-1240-EN (Ceasefire)
   ✓ Extracted 4 speeches

📄 Processing S_PV-1241-EN (Ceasefire)
   ✓ Extracted 4 speeches

📄 Processing S_PV-1242-EN (Ceasefire)
   ✓ Extracted 17 speeches

📄 Processing S_PV-1244-EN (Ceasefire)
   ✓ Extracted 2 speeches

📄 Processing S_PV-1245-EN (Ceasefire)
   ✓ Extracted 8 speeches

📄 Processing S_PV-1247-EN (Ceasefire)
   ✓ Extracted 14 speeches

📄 Processing S_PV-1248-EN (Ceasefire)
   ✓ Extracted 2 speeches

📄 Processing S_PV-1

In [ ]:
def create_training_dataframe(processed_meetings):
    """
    Convert processed meetings (speeches only) into a structured DataFrame.
    Each row = one speech from one meeting.
    """

    rows = []

    for meeting_id, meeting_data in processed_meetings.items():
        speeches = meeting_data.get("speeches", [])

        for sp in speeches:
            row = {
                "meeting_id": meeting_id,
                "topic": meeting_data.get("topic"),
                "pdf_path": meeting_data.get("pdf_path"),

                # From each speech dict
                "speaker": sp.get("speaker"),
                "country": sp.get("country"),
                "speech_text": sp.get("text"),

                # Useful NLP metadata
                "speech_length_chars": len(sp.get("text", "")),
                "speech_length_words": len(sp.get("text", "").split()),
            }

            rows.append(row)

    return pd.DataFrame(rows)


# Create the speeches dataframe
training_df = create_training_dataframe(processed_meetings)

print("\n" + "="*70)
print("FINAL SPEECH DATASET")
print("="*70)
print(f"Total speeches extracted: {len(training_df)}")
print(f"Dataset shape: {training_df.shape}")

print("\nColumns:")
print(list(training_df.columns))

print("\nTopic distribution:")
print(training_df['topic'].value_counts())

print("\nSample rows:")
print(training_df[['meeting_id', 'speaker', 'country', 'speech_length_words']].head(10))

# Save to CSV
output_path = "/content/un_speeches_dataset.csv"
training_df.to_csv(output_path, index=False)
print(f"\n✓ Saved speeches dataset to: {output_path}")


In [ ]:
# Get unique countries and their counts
unique_countries = training_df['country'].value_counts()

print("Unique Countries and their counts in the dataset:")
print(unique_countries.to_string())

print(f"\nTotal number of unique countries: {len(unique_countries)}")

In [ ]:
def clean_text(text):
    """
    Clean and normalize text.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-z0-9\s\.\,\!\?]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning
print("Cleaning speech text...")
training_df['speech_clean'] = training_df['speech_text'].apply(clean_text)


In [ ]:
def extract_sentiment_features(text):
    """
    Extract sentiment scores using VADER.
    """
    if not text:
        return {'sentiment_neg': 0, 'sentiment_neu': 0, 'sentiment_pos': 0, 'sentiment_compound': 0}

    sia = SentimentIntensityAnalyzer()
    scores = sia.polarity_scores(text)

    return {
        'sentiment_neg': scores['neg'],
        'sentiment_neu': scores['neu'],
        'sentiment_pos': scores['pos'],
        'sentiment_compound': scores['compound']
    }

print("Extracting sentiment features...")
sentiment_df = training_df['speech_clean'].apply(extract_sentiment_features).apply(pd.Series)
df = pd.concat([training_df, sentiment_df], axis=1)

print("\nSentiment statistics:")
print(sentiment_df.describe())

In [ ]:
def extract_keyword_features(text):
    """
    Extract presence of important keywords
    """
    if not text:
        return {f'kw_{k}': 0 for k in ['condemn', 'support', 'oppose', 'sanction', 'peace', 'security', 'humanitarian', 'violation']}

    text_lower = text.lower()

    keywords = {
        'condemn': ['condemn', 'condemnation', 'denounce'],
        'support': ['support', 'agree', 'favor', 'backing'],
        'oppose': ['oppose', 'against', 'reject', 'refuse'],
        'sanction': ['sanction', 'embargo', 'restriction'],
        'peace': ['peace', 'peaceful', 'stability'],
        'security': ['security', 'threat', 'danger'],
        'humanitarian': ['humanitarian', 'civilian', 'victim'],
        'violation': ['violation', 'breach', 'illegal']
    }

    features = {}
    for key, terms in keywords.items():
        features[f'kw_{key}'] = int(any(term in text_lower for term in terms))

    return features

print("Extracting keyword features...")
keyword_df = df['speech_clean'].apply(extract_keyword_features).apply(pd.Series)
df = pd.concat([df, keyword_df], axis=1)

## Stance Prediction


In [ ]:
print("="*70)
print("STRATEGY 1: STANCE DETECTION")
print("="*70)

# Define stance categories
stance_categories = {
    'Strong Support': {
        'description': 'Strongly condemns violation, calls for immediate action, supports sanctions',
        'keywords': ['condemn', 'violation', 'illegal', 'unacceptable', 'immediate', 'must', 'demand']
    },
    'Moderate Support': {
        'description': 'Agrees with concerns but with reservations or calls for balanced approach',
        'keywords': ['support', 'agree', 'concern', 'however', 'while', 'balance', 'consider']
    },
    'Neutral': {
        'description': 'Calls for dialogue, emphasizes both sides, procedural language',
        'keywords': ['dialogue', 'negotiation', 'both', 'all parties', 'peaceful', 'diplomatic']
    },
    'Cautious Opposition': {
        'description': 'Questions approach, emphasizes sovereignty, suggests alternatives',
        'keywords': ['sovereignty', 'internal affairs', 'interference', 'alternative', 'concern about']
    },
    'Strong Opposition': {
        'description': 'Rejects resolution, defends accused party, criticizes approach',
        'keywords': ['reject', 'oppose', 'unjust', 'illegitimate', 'unilateral', 'violation of sovereignty']
    }
}

print("\nStance Categories Defined:")

In [ ]:
def detect_stance_rule_based(speech_text):
    """
    Detect stance using a rule-based approach based ONLY on speech text.

    Args:
        speech_text (str): The speech text for one speaker.

    Returns:
        stance (str): One of the stance categories.
    """

    # If no speech available
    if not speech_text or pd.isna(speech_text):
        return "Unknown"

    speech_lower = speech_text.lower()

    # Count keyword hits for each stance category
    stance_scores = {}

    for stance_name, stance_info in stance_categories.items():
        score = sum(
            1 for keyword in stance_info['keywords']
            if keyword in speech_lower
        )
        stance_scores[stance_name] = score

    # Determine best stance
    max_score = max(stance_scores.values())

    if max_score == 0:
        # No keywords found → cannot determine stance
        return "Unknown"

    # Return stance with highest score
    top_stance = max(stance_scores, key=stance_scores.get)
    return top_stance


In [ ]:
#Apply stance detection
print("\n" + "="*70)
print("Applying Rule-Based Stance Detection...")
print("="*70)

df['stance'] = df.apply(lambda row: detect_stance_rule_based(
    row['speech_clean']
), axis=1)

# Show distribution
print("\nStance Distribution:")
print(df['stance'].value_counts())

In [ ]:
def extract_stance_features(speech_text):
    """
    Extract numerical features that indicate stance intensity and type.
    """
    if not speech_text or pd.isna(speech_text):
        return {
            'stance_intensity': 0,
            'stance_support_score': 0,
            'stance_opposition_score': 0,
            'stance_neutrality_score': 0,
            'stance_certainty_score': 0
        }

    speech_lower = speech_text.lower()
    word_count = len(speech_text.split())

    # Support indicators
    support_words = ['support', 'agree', 'favor', 'endorse', 'condemn', 'violation']
    support_score = sum(speech_lower.count(w) for w in support_words) / max(word_count, 1)

    # Opposition indicators
    opposition_words = ['oppose', 'reject', 'against', 'disagree', 'sovereignty', 'interference']
    opposition_score = sum(speech_lower.count(w) for w in opposition_words) / max(word_count, 1)

    # Neutrality indicators
    neutral_words = ['dialogue', 'negotiation', 'peaceful', 'both', 'all parties', 'balance']
    neutrality_score = sum(speech_lower.count(w) for w in neutral_words) / max(word_count, 1)

    # Certainty/intensity indicators
    strong_words = ['must', 'immediately', 'urgent', 'demand', 'insist', 'strongly']
    certainty_score = sum(speech_lower.count(w) for w in strong_words) / max(word_count, 1)

    # Overall intensity (how strong the stance is)
    intensity = support_score + opposition_score + certainty_score

    return {
        'stance_intensity': intensity,
        'stance_support_score': support_score,
        'stance_opposition_score': opposition_score,
        'stance_neutrality_score': neutrality_score,
        'stance_certainty_score': certainty_score
    }


print("\nExtracting Advanced Stance Features...")

stance_features_df = df['speech_clean'].apply(extract_stance_features).apply(pd.Series)
df = pd.concat([df, stance_features_df], axis=1)

print("\n✓ Stance features added!")
print("\nStance Feature Statistics:")
print(stance_features_df.describe())


In [ ]:
print("\n" + "="*70)
print("STANCE ANALYSIS INSIGHTS")
print("="*70)

# Normalize country names to handle variations
df['country_normalized'] = df['country'].str.lower().str.strip()

# Define P5 members (Permanent members of the UN Security Council)
p5_members = [
    'united states of america', 'united states', 'united kingdom',
    'china', 'russian federation', 'france'
]
df['is_p5'] = df['country_normalized'].isin(p5_members).astype(int)

# Define a list of Western countries for categorization
western_countries = [
    'united states of america', 'united states', 'united kingdom', 'canada',
    'australia', 'new zealand', 'ireland', 'norway', 'germany', 'italy',
    'sweden', 'netherlands', 'denmark', 'austria', 'finland', 'greece',
    'belgium', 'portugal', 'spain', 'luxembourg', 'iceland', 'switzerland',
    'malta', 'estonia', 'latvia', 'lithuania', 'slovenia', 'croatia',
    'czech republic', 'slovakia', 'poland', 'japan', 'republic of korea'
]
df['is_western'] = df['country_normalized'].isin(western_countries).astype(int)

# Determine if a country actually spoke (speech_length_words > 0)
df['country_spoke'] = (df['speech_length_words'] > 0).astype(int)

# 1. Countries with most varied stances
print("\n1. Countries with Most Diverse Stances:")
country_stance_variety = df.groupby('country')['stance'].nunique().sort_values(ascending=False)
print(country_stance_variety.head(10))

# 2. Most common stance by country group
print("\n2. Stance Patterns by Country Group:")
print("\nWestern Countries:")
print(df[df['is_western'] == 1]['stance'].value_counts())

print("\nNon-Western Countries:")
print(df[df['is_western'] == 0]['stance'].value_counts())

print("\nP5 Members:")
print(df[df['is_p5'] == 1]['stance'].value_counts())

# 4. Countries that spoke vs didn't speak
print("\n4. Stance Distribution: Spoke vs Didn't Speak:")
print("\nCountries that spoke:")
print(df[df['country_spoke'] == 1]['stance'].value_counts())

print("\nCountries that didn't speak:")
print(df[df['country_spoke'] == 0]['stance'].value_counts())

# 5. Example speeches for each stance
print("\n" + "="*70)
print("5. Example Speeches by Stance")
print("="*70)

for stance in df['stance'].unique():
    examples = df[df['stance'] == stance][df['country_spoke'] == 1]
    if len(examples) > 0:
        example = examples.iloc[0]
        print(f"\n{stance}:")
        print(f"  Country: {example['country']}")
        print(f"  Speech preview: {example['speech_clean'][:200]}...")

##  RHETORICAL STRATEGY


In [ ]:
print("="*70)
print("STRATEGY 2: RHETORICAL STRATEGY ANALYSIS")
print("="*70)

# Define rhetorical strategies with indicators
rhetorical_strategies = {
    'Appeal to Authority': {
        'description': 'References to UN Charter, international law, resolutions',
        'indicators': [
            'charter', 'article', 'resolution', 'international law',
            'united nations', 'security council', 'general assembly',
            'convention', 'treaty', 'protocol'
        ]
    },

    'Moral Framing': {
        'description': 'Use of moral and ethical language',
        'indicators': [
            'right', 'wrong', 'justice', 'injustice', 'moral', 'ethical',
            'principle', 'values', 'conscience', 'duty', 'obligation',
            'good', 'evil', 'legitimate', 'illegitimate'
        ]
    },

    'Emotional Appeal': {
        'description': 'Appeals to emotions through humanitarian language',
        'indicators': [
            'victim', 'suffer', 'tragedy', 'crisis', 'humanitarian',
            'innocent', 'civilian', 'children', 'families', 'devastating',
            'terrible', 'horrific', 'tragic', 'catastrophe'
        ]
    },

    'Urgency Framing': {
        'description': 'Emphasis on immediate action and time sensitivity',
        'indicators': [
            'immediate', 'immediately', 'urgent', 'urgently', 'now',
            'must', 'cannot wait', 'without delay', 'as soon as possible',
            'time is running', 'critical', 'pressing'
        ]
    },

    'Coalition Building': {
        'description': 'References to international consensus and unity',
        'indicators': [
            'international community', 'world', 'all states', 'together',
            'collective', 'unanimous', 'consensus', 'solidarity',
            'common', 'united', 'cooperation', 'partnership'
        ]
    },

    'Historical Precedent': {
        'description': 'References to past events, conflicts, or decisions',
        'indicators': [
            'history', 'past', 'previous', 'before', 'precedent',
            'remember', 'learned', 'experience', 'tradition',
            'always', 'never', 'repeatedly'
        ]
    },

    'Sovereignty Defense': {
        'description': 'Emphasis on non-interference and territorial integrity',
        'indicators': [
            'sovereignty', 'territorial integrity', 'internal affairs',
            'interference', 'intervention', 'respect', 'independence',
            'self-determination', 'domestic', 'non-interference'
        ]
    },

    'Diplomatic Hedging': {
        'description': 'Cautious language that softens positions',
        'indicators': [
            'however', 'although', 'while', 'nevertheless', 'nonetheless',
            'perhaps', 'may', 'might', 'could', 'would', 'possibly',
            'consider', 'suggest', 'appear', 'seem'
        ]
    },

    'Assertive Language': {
        'description': 'Strong, definitive statements and demands',
        'indicators': [
            'must', 'demand', 'require', 'insist', 'will', 'shall',
            'condemn', 'reject', 'unacceptable', 'strongly', 'firmly',
            'clearly', 'categorically', 'absolutely'
        ]
    },

    'Peace/Dialogue Emphasis': {
        'description': 'Focus on peaceful resolution and negotiation',
        'indicators': [
            'peace', 'peaceful', 'dialogue', 'negotiation', 'diplomatic',
            'talks', 'mediation', 'compromise', 'resolution', 'settlement',
            'agreement', 'understanding', 'reconciliation'
        ]
    }
}

print("\nRhetorical Strategies Defined:")
print(f"Total strategies: {len(rhetorical_strategies)}\n")

In [ ]:
def extract_rhetorical_features(speech_text):
    """
    Extract all rhetorical strategy features from a speech.

    Returns:
        Dictionary with counts and presence of each strategy
    """
    if not speech_text or pd.isna(speech_text):
        # Return zeros for all strategies
        features = {}
        for strategy in rhetorical_strategies.keys():
            strategy_key = strategy.lower().replace(' ', '_').replace('/', '_')
            features[f'rhet_{strategy_key}_count'] = 0
            features[f'rhet_{strategy_key}_present'] = 0
        features['rhet_total_strategies'] = 0
        features['rhet_diversity'] = 0
        return features

    speech_lower = speech_text.lower()
    word_count = len(speech_text.split())

    features = {}
    strategies_used = 0

    for strategy_name, strategy_info in rhetorical_strategies.items():
        # Create clean key name
        strategy_key = strategy_name.lower().replace(' ', '_').replace('/', '_')

        # Count occurrences of indicators
        count = sum(speech_lower.count(indicator) for indicator in strategy_info['indicators'])

        # Normalize by word count (per 100 words)
        normalized_count = (count / max(word_count, 1)) * 100

        # Store features
        features[f'rhet_{strategy_key}_count'] = normalized_count
        features[f'rhet_{strategy_key}_present'] = 1 if count > 0 else 0

        if count > 0:
            strategies_used += 1

    # Overall metrics
    features['rhet_total_strategies'] = strategies_used
    features['rhet_diversity'] = strategies_used / len(rhetorical_strategies)  # Proportion of strategies used

    return features


print("\n" + "="*70)
print("Extracting Rhetorical Features...")
print("="*70)

rhetorical_features_df = df['speech_clean'].apply(extract_rhetorical_features).apply(pd.Series)
df = pd.concat([df, rhetorical_features_df], axis=1)

print(f"\n✓ Added {len(rhetorical_features_df.columns)} rhetorical feature columns")

# Show summary statistics
print("\nRhetorical Strategy Usage Statistics (for speeches that were given):")
speeches_only = df[df['country_spoke'] == 1]

strategy_usage = {}
for strategy in rhetorical_strategies.keys():
    strategy_key = strategy.lower().replace(' ', '_').replace('/', '_')
    col_name = f'rhet_{strategy_key}_present'
    usage_pct = speeches_only[col_name].mean() * 100
    avg_count = speeches_only[f'rhet_{strategy_key}_count'].mean()
    strategy_usage[strategy] = {'usage_pct': usage_pct, 'avg_count': avg_count}

# Sort by usage
strategy_usage_sorted = sorted(strategy_usage.items(), key=lambda x: x[1]['usage_pct'], reverse=True)

print("\nStrategy Usage Ranking:")
for i, (strategy, stats) in enumerate(strategy_usage_sorted, 1):
    print(f"{i:2d}. {strategy:30s} - Used in {stats['usage_pct']:5.1f}% of speeches (avg: {stats['avg_count']:.2f} per 100 words)")

In [ ]:
print("\n" + "="*70)
print("RHETORICAL PATTERN ANALYSIS")
print("="*70)

speeches_only = df[df['country_spoke'] == 1].copy()

# 1. Average number of strategies used
print("\n1. Rhetorical Strategy Diversity:")
print(f"   Average strategies used per speech: {speeches_only['rhet_total_strategies'].mean():.2f}")
print(f"   Max strategies in a single speech: {speeches_only['rhet_total_strategies'].max():.0f}")
print(f"   Min strategies in a single speech: {speeches_only['rhet_total_strategies'].min():.0f}")

# 3. Most common strategy combinations
print("\n3. Top Strategy Combinations:")

# Get presence columns
presence_cols = [col for col in df.columns if col.startswith('rhet_') and col.endswith('_present')]

# Create strategy combination signatures
speeches_only['strategy_signature'] = speeches_only[presence_cols].apply(
    lambda row: ','.join([col.replace('rhet_', '').replace('_present', '')
                         for col in presence_cols if row[col] == 1]),
    axis=1
)

top_combinations = speeches_only['strategy_signature'].value_counts().head(10)
print("\nMost common strategy combinations:")
for i, (combo, count) in enumerate(top_combinations.items(), 1):
    strategies = combo.split(',') if combo else ['None']
    print(f"{i:2d}. ({count:2d} speeches) {', '.join(strategies[:3])}{'...' if len(strategies) > 3 else ''}")

# 4. Strategies by country group
print("\n4. Rhetorical Strategy Use by Country Group:")

print("\nWestern Countries:")
western_strategies = speeches_only[speeches_only['is_western'] == 1]['rhet_total_strategies'].describe()
print(f"   Mean: {western_strategies['mean']:.2f}, Std: {western_strategies['std']:.2f}")

print("\nNon-Western Countries:")
non_western_strategies = speeches_only[speeches_only['is_western'] == 0]['rhet_total_strategies'].describe()
print(f"   Mean: {non_western_strategies['mean']:.2f}, Std: {non_western_strategies['std']:.2f}")

print("\nP5 Members:")
p5_strategies = speeches_only[speeches_only['is_p5'] == 1]['rhet_total_strategies'].describe()
print(f"   Mean: {p5_strategies['mean']:.2f}, Std: {p5_strategies['std']:.2f}")

print("\nNon-P5 Members:")
non_p5_strategies = speeches_only[speeches_only['is_p5'] == 0]['rhet_total_strategies'].describe()
print(f"   Mean: {non_p5_strategies['mean']:.2f}, Std: {non_p5_strategies['std']:.2f}")

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# PLOT 1 — Top 10 Rhetorical Strategies
ax1 = axes[0, 0]

strategy_presence_cols = [
    col for col in df.columns
    if col.startswith('rhet_') and col.endswith('_present')
]

# Convert column names to human-readable format
strategy_names = [
    col.replace('rhet_', '')
       .replace('_present', '')
       .replace('_', ' ')
       .title()
    for col in strategy_presence_cols
]

# Compute overall usage (% of speeches using each strategy)
overall_usage = speeches_only[strategy_presence_cols].mean() * 100
overall_usage.index = strategy_names

# Top 10
top_10_strategies = overall_usage.sort_values(ascending=True).tail(10)

# Plot
top_10_strategies.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Usage (%)')
ax1.set_title('Top 10 Most Used Rhetorical Strategies', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# PLOT 2 — Strategy Diversity Distribution (Overall Histogram)
ax2 = axes[0, 1]

ax2.hist(
    speeches_only['rhet_total_strategies'],
    bins=range(0, 12),
    alpha=0.75,
    color='slateblue'
)

ax2.set_xlabel('Number of Strategies Used')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Strategy Diversity', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# PLOT 3 — Strategy Diversity by Country Group

ax3 = axes[1, 0]

box_data = []
box_labels = []

groups = [
    ('Western',      speeches_only['is_western'] == 1),
    ('Non-Western',  speeches_only['is_western'] == 0),
    ('P5',           speeches_only['is_p5'] == 1),
    ('Non-P5',       speeches_only['is_p5'] == 0)
]

for group_name, mask in groups:
    data = speeches_only[mask]['rhet_total_strategies'].values
    box_data.append(data)
    box_labels.append(group_name)

bp = ax3.boxplot(box_data, labels=box_labels, patch_artist=True)

# Color styling
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax3.set_ylabel('Number of Strategies Used')
ax3.set_title('Rhetorical Strategy Diversity by Country Group', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# PLOT 4 — Heatmap of Strategy Usage (%)

ax4 = axes[1, 1]

usage = speeches_only[strategy_presence_cols].mean() * 100
usage.index = strategy_names

sns.heatmap(
    usage.to_frame().T,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    ax=ax4,
    cbar_kws={'label': 'Usage %'}
)

ax4.set_title('Overall Rhetorical Strategy Usage (%)', fontsize=14, fontweight='bold')
ax4.set_ylabel('')
ax4.set_xlabel('Strategies')

plt.tight_layout()
plt.show()


## TOPIC MODELLING

In [ ]:
speeches_only.head()

In [ ]:
print("="*70)
print("STRATEGY 3: TOPIC MODELING & THEMATIC ANALYSIS")
print("="*70)

print(f"\nDataset Info:")
print(f"  Total rows in dataset: {len(df)}")
print(f"  Speeches given: {len(speeches_only)}")
print(f"  Average speech length: {speeches_only['speech_length_words'].mean():.0f} characters")

# Prepare text corpus
corpus = speeches_only['speech_clean'].tolist()

print(f"\nCorpus prepared: {len(corpus)} speeches")

# Additional text preprocessing for topic modeling
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def preprocess_for_topics(text):
    """
    Additional preprocessing specifically for topic modeling.
    """
    if not text or pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove very short words (less than 3 characters)
    text = re.sub(r'\b\w{1,2}\b', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply preprocessing
corpus_processed = [preprocess_for_topics(text) for text in corpus]

# Remove empty documents
valid_indices = [i for i, text in enumerate(corpus_processed) if len(text) > 50]
corpus_processed = [corpus_processed[i] for i in valid_indices]
speeches_for_topics = speeches_only.iloc[valid_indices].copy()

print(f"After filtering: {len(corpus_processed)} valid speeches for topic modeling")

In [ ]:
print("\n" + "="*70)
print("LDA TOPIC MODELING")
print("="*70)

# Create document-term matrix
print("\nCreating document-term matrix...")

vectorizer = CountVectorizer(
    max_features=500,           # Top 500 words
    min_df=2,                   # Word must appear in at least 2 documents
    max_df=0.8,                 # Word must appear in less than 80% of documents
    stop_words='english',
    ngram_range=(1, 2)          # Unigrams and bigrams
)

doc_term_matrix = vectorizer.fit_transform(corpus_processed)
feature_names = vectorizer.get_feature_names_out()

print(f"✓ Document-term matrix: {doc_term_matrix.shape}")
print(f"  Documents: {doc_term_matrix.shape[0]}")
print(f"  Features: {doc_term_matrix.shape[1]}")

# Train LDA model
print("\nTraining LDA model...")

n_topics = 5  # We'll try 5 topics

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=50,
    learning_method='online',
    random_state=42,
    n_jobs=-1
)

lda_topics = lda_model.fit_transform(doc_term_matrix)

print(f"✓ LDA model trained with {n_topics} topics")

# Display topics
print("\n" + "="*70)
print("DISCOVERED TOPICS (LDA)")
print("="*70)

def display_topics(model, feature_names, n_top_words=10):
    """Display top words for each topic"""
    topics_summary = []

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        top_weights = [topic[i] for i in top_indices]

        topics_summary.append({
            'topic_id': topic_idx,
            'top_words': top_words,
            'weights': top_weights
        })

        print(f"\nTopic {topic_idx}:")
        print(f"  Top words: {', '.join(top_words[:10])}")

    return topics_summary

lda_topics_summary = display_topics(lda_model, feature_names, n_top_words=10)

# Assign dominant topic to each speech
speeches_for_topics['lda_topic'] = lda_topics.argmax(axis=1)
speeches_for_topics['lda_topic_weight'] = lda_topics.max(axis=1)

print(f"\n✓ Assigned dominant topics to speeches")

In [ ]:
topic_names = {
    0: "Sovereignty & Non-Interference",
    1: "Humanitarian Crisis & Protection",
    2: "International Law & Charter Violations",
    3: "Peaceful Resolution & Dialogue",
    4: "Security Threats & Sanctions"
}

# Let's analyze each topic to suggest names
for topic_id in range(n_topics):
    print(f"\n{'-'*70}")
    print(f"Topic {topic_id} Analysis:")
    print(f"{'-'*70}")

    # Get speeches for this topic
    topic_speeches = speeches_for_topics[speeches_for_topics['lda_topic'] == topic_id]

    print(f"Number of speeches: {len(topic_speeches)}")


    # Country distribution
    print(f"\nTop countries:")
    print(topic_speeches['country'].value_counts().head(5))

    # Topic distribution
    print(f"\nTopic distribution:")
    print(topic_speeches['topic'].value_counts())

    # Example speech
    if len(topic_speeches) > 0:
        example = topic_speeches.iloc[0]
        print(f"\nExample speech:")
        print(f"  Country: {example['country']}")
        print(f"  Preview: {example['speech_clean'][:200]}...")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Topic distribution
ax1 = axes[0, 0]
topic_counts = speeches_for_topics['lda_topic'].value_counts().sort_index()
ax1.bar(topic_counts.index, topic_counts.values, color='steelblue')
ax1.set_xlabel('Topic ID')
ax1.set_ylabel('Number of Speeches')
ax1.set_title('Distribution of Speeches Across Topics (LDA)', fontsize=14, fontweight='bold')
ax1.set_xticks(range(n_topics))


# Plot 3: Topic by meeting topic
ax3 = axes[0, 1]
topic_meeting_crosstab = pd.crosstab(speeches_for_topics['lda_topic'],
                                     speeches_for_topics['topic'])
topic_meeting_crosstab.plot(kind='bar', ax=ax3, stacked=True)
ax3.set_xlabel('Topic ID')
ax3.set_ylabel('Count')
ax3.set_title('LDA Topic Distribution by Meeting Type', fontsize=14, fontweight='bold')
ax3.legend(title='Meeting Type', bbox_to_anchor=(1.05, 1))
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

# Plot 4: Topic weight distribution
ax4 = axes[1, 0]
for topic_id in range(n_topics):
    topic_weights = speeches_for_topics[speeches_for_topics['lda_topic'] == topic_id]['lda_topic_weight']
    ax4.hist(topic_weights, alpha=0.6, label=f'Topic {topic_id}', bins=20)

ax4.set_xlabel('Topic Weight (Confidence)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Topic Assignment Confidence', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n" + "="*70)
print("TOPIC STATISTICS")
print("="*70)

print(f"\nAverage topic weight: {speeches_for_topics['lda_topic_weight'].mean():.3f}")
print(f"Min topic weight: {speeches_for_topics['lda_topic_weight'].min():.3f}")
print(f"Max topic weight: {speeches_for_topics['lda_topic_weight'].max():.3f}")

print(f"\nSpeeches with high confidence (weight > 0.5): {(speeches_for_topics['lda_topic_weight'] > 0.5).sum()}")
print(f"Speeches with low confidence (weight < 0.3): {(speeches_for_topics['lda_topic_weight'] < 0.3).sum()}")

In [ ]:
print("\n" + "="*70)
print("BERTOPIC MODELING (Advanced)")
print("="*70)

# Install BERTopic if not already installed
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    print("✓ BERTopic available")
    bertopic_available = True
except ImportError:
    print("⚠️  BERTopic not installed. Installing now...")
    print("Run: !pip install bertopic")
    bertopic_available = False

if bertopic_available:
    print("\nTraining BERTopic model...")
    print("(This may take a few minutes...)")

    # Initialize BERTopic
    topic_model = BERTopic(
        language="english",
        calculate_probabilities=True,
        verbose=True,
        min_topic_size=3  # Minimum speeches per topic
    )

    # Fit the model
    topics, probabilities = topic_model.fit_transform(corpus_processed)

    print(f"\n✓ BERTopic model trained")
    print(f"  Number of topics found: {len(set(topics)) - 1}")  # -1 for outlier topic

    # Get topic info
    topic_info = topic_model.get_topic_info()

    print("\n" + "="*70)
    print("BERTOPIC DISCOVERED TOPICS")
    print("="*70)

    print(topic_info[['Topic', 'Count', 'Name']])

    # Show top words for each topic
    print("\nTop words per topic:")
    for topic_id in topic_info['Topic']:
        if topic_id == -1:  # Skip outlier topic
            continue
        topic_words = topic_model.get_topic(topic_id)
        if topic_words:
            words = [word for word, _ in topic_words[:10]]
            print(f"\nTopic {topic_id}: {', '.join(words)}")

    # Assign to dataframe
    speeches_for_topics['bertopic_topic'] = topics
    speeches_for_topics['bertopic_prob'] = probabilities.max(axis=1) if len(probabilities.shape) > 1 else probabilities

    # Visualizations
    print("\nGenerating BERTopic visualizations...")

    # 1. Topic visualization
    try:
        fig1 = topic_model.visualize_topics()
        fig1.show()
    except:
        print("  Note: Interactive topic visualization requires plotly")

    # 2. Topic barchart
    try:
        fig2 = topic_model.visualize_barchart(top_n_topics=10)
        fig2.show()
    except:
        print("  Note: Barchart visualization requires plotly")

else:
    print("\n  Skipping BERTopic analysis")
    print("To enable, run: !pip install bertopic sentence-transformers")


## Framing Analysis

In [ ]:
print("="*70)
print("FRAMING ANALYSIS")
print("="*70)

# Define comprehensive framing categories
framing_categories = {
    'Aggression Frame': {
        'description': 'Frames the situation as an act of aggression/attack',
        'keywords': [
            'invasion', 'invade', 'attack', 'aggression', 'aggressive',
            'assault', 'hostile', 'hostility', 'military action', 'force',
            'occupation', 'occupy', 'incursion', 'raid'
        ],
        'phrases': [
            'armed attack', 'military invasion', 'act of aggression',
            'hostile action', 'use of force'
        ]
    },

    'Legal/Charter Violation Frame': {
        'description': 'Frames as violation of international law and UN Charter',
        'keywords': [
            'violation', 'violate', 'breach', 'illegal', 'unlawful',
            'charter', 'article', 'law', 'legal', 'unlawfully',
            'contravene', 'infringement', 'transgression'
        ],
        'phrases': [
            'violation of the charter', 'breach of international law',
            'illegal act', 'contravention of', 'article 2'
        ]
    },

    'Humanitarian/Human Rights Frame': {
        'description': 'Frames in terms of human suffering and rights',
        'keywords': [
            'humanitarian', 'civilian', 'civilians', 'victim', 'victims',
            'suffering', 'human rights', 'casualties', 'innocent',
            'population', 'people', 'lives', 'deaths', 'wounded'
        ],
        'phrases': [
            'human rights', 'civilian casualties', 'innocent victims',
            'humanitarian crisis', 'human suffering'
        ]
    },

    'Security Threat Frame': {
        'description': 'Frames as threat to regional/international security',
        'keywords': [
            'threat', 'threaten', 'security', 'peace', 'stability',
            'danger', 'dangerous', 'risk', 'jeopardize', 'endanger',
            'destabilize', 'destabilization', 'menace'
        ],
        'phrases': [
            'threat to peace', 'international security', 'regional stability',
            'breach of peace', 'threat to security'
        ]
    },

    'Sovereignty/Territorial Frame': {
        'description': 'Frames as violation of sovereignty and territorial integrity',
        'keywords': [
            'sovereignty', 'sovereign', 'territorial integrity', 'territory',
            'independence', 'independent', 'borders', 'frontier',
            'self-determination', 'autonomy', 'national'
        ],
        'phrases': [
            'territorial integrity', 'sovereign state', 'sovereign rights',
            'national sovereignty', 'independent state'
        ]
    },

    'Diplomatic/Dialogue Frame': {
        'description': 'Frames as situation requiring diplomatic solution',
        'keywords': [
            'dialogue', 'negotiation', 'negotiate', 'diplomatic', 'diplomacy',
            'peaceful', 'peaceful resolution', 'talks', 'discussion',
            'mediation', 'compromise', 'settlement', 'consultation'
        ],
        'phrases': [
            'peaceful resolution', 'diplomatic solution', 'through dialogue',
            'negotiated settlement', 'peaceful means'
        ]
    },

    'Self-Defense Frame': {
        'description': 'Frames actions as legitimate self-defense',
        'keywords': [
            'self-defense', 'defense', 'defend', 'protection', 'protect',
            'security concerns', 'defending', 'defensive', 'legitimate',
            'safeguard', 'shield'
        ],
        'phrases': [
            'right to self-defense', 'legitimate defense', 'defensive action',
            'protecting interests', 'security interests'
        ]
    },

    'Non-Interference Frame': {
        'description': 'Frames as issue of non-interference in internal affairs',
        'keywords': [
            'interference', 'intervene', 'intervention', 'internal affairs',
            'domestic', 'non-interference', 'meddling', 'intrusion',
            'respect', 'sovereign affairs'
        ],
        'phrases': [
            'internal affairs', 'non-interference', 'domestic matters',
            'foreign intervention', 'respect for sovereignty'
        ]
    },

    'Justice/Accountability Frame': {
        'description': 'Frames in terms of justice and accountability',
        'keywords': [
            'justice', 'accountability', 'responsible', 'responsibility',
            'accountable', 'consequences', 'impunity', 'guilty',
            'perpetrator', 'punishment', 'sanctions'
        ],
        'phrases': [
            'held accountable', 'face consequences', 'bring to justice',
            'end impunity', 'must answer'
        ]
    },

    'Crisis/Emergency Frame': {
        'description': 'Frames as urgent crisis requiring immediate action',
        'keywords': [
            'crisis', 'emergency', 'urgent', 'urgently', 'immediate',
            'immediately', 'critical', 'pressing', 'grave', 'serious',
            'catastrophe', 'disaster'
        ],
        'phrases': [
            'grave crisis', 'urgent situation', 'immediate action',
            'critical moment', 'emergency measures'
        ]
    }
}

print(f"\nFraming Categories Defined: {len(framing_categories)}")
print("\nCategories:")
for frame_name, frame_info in framing_categories.items():
    print(f"  • {frame_name}")
    print(f"    {frame_info['description']}")

In [ ]:
def extract_framing_features(speech_text):
    """
    Extract framing features from speech text.
    Returns counts, presence, and intensity for each frame.
    """
    if not speech_text or pd.isna(speech_text):
        # Return zeros for all frames
        features = {}
        for frame_name in framing_categories.keys():
            frame_key = frame_name.lower().replace('/', '_').replace(' ', '_')
            features[f'frame_{frame_key}_count'] = 0
            features[f'frame_{frame_key}_present'] = 0
            features[f'frame_{frame_key}_phrase_count'] = 0
        features['frame_total_count'] = 0
        features['frame_diversity'] = 0
        features['frame_dominant'] = 'None'
        return features

    speech_lower = speech_text.lower()
    word_count = len(speech_text.split())

    features = {}
    frame_scores = {}
    frames_present = 0

    for frame_name, frame_info in framing_categories.items():
        frame_key = frame_name.lower().replace('/', '_').replace(' ', '_')

        # Count keyword occurrences
        keyword_count = sum(speech_lower.count(keyword) for keyword in frame_info['keywords'])

        # Count phrase occurrences (weighted more heavily)
        phrase_count = sum(speech_lower.count(phrase) for phrase in frame_info['phrases'])

        # Total score (phrases count double)
        total_score = keyword_count + (phrase_count * 2)

        # Normalize by word count (per 100 words)
        normalized_score = (total_score / max(word_count, 1)) * 100

        # Store features
        features[f'frame_{frame_key}_count'] = normalized_score
        features[f'frame_{frame_key}_present'] = 1 if total_score > 0 else 0
        features[f'frame_{frame_key}_phrase_count'] = phrase_count

        frame_scores[frame_name] = normalized_score

        if total_score > 0:
            frames_present += 1

    # Overall metrics
    features['frame_total_count'] = sum(frame_scores.values())
    features['frame_diversity'] = frames_present

    # Dominant frame
    if frame_scores and max(frame_scores.values()) > 0:
        features['frame_dominant'] = max(frame_scores, key=frame_scores.get)
    else:
        features['frame_dominant'] = 'None'

    return features


print("\n" + "="*70)
print("Extracting Framing Features...")
print("="*70)

framing_features_df = df['speech_clean'].apply(extract_framing_features).apply(pd.Series)
df = pd.concat([df, framing_features_df], axis=1)

print(f"\n✓ Added {len(framing_features_df.columns)} framing feature columns")

# Show summary for speeches only
speeches_only = df[df['country_spoke'] == 1]

print("\nFraming Feature Statistics (for speeches given):")
print(f"  Average frames per speech: {speeches_only['frame_diversity'].mean():.2f}")
print(f"  Average framing intensity: {speeches_only['frame_total_count'].mean():.2f}")

# Show frame usage
print("\nFrame Usage Frequency:")
for frame_name in framing_categories.keys():
    frame_key = frame_name.lower().replace('/', '_').replace(' ', '_')
    usage_pct = speeches_only[f'frame_{frame_key}_present'].mean() * 100
    print(f"  {frame_name:35s}: {usage_pct:5.1f}%")

In [ ]:
print("\n" + "="*70)
print("DOMINANT FRAME ANALYSIS")
print("="*70)

speeches_only = df[df['country_spoke'] == 1].copy()

# Overall distribution
print("\n1. Dominant Frame Distribution:")
dominant_frames = speeches_only['frame_dominant'].value_counts()
print(dominant_frames)

# Dominant frame by topic
print("\n3. Dominant Frame by Meeting Topic:")
frame_topic_crosstab = pd.crosstab(
    speeches_only['topic'],
    speeches_only['frame_dominant']
)
print(frame_topic_crosstab)

# Dominant frame by country group
print("\n4. Dominant Frame by Country Group:")

print("\nWestern Countries:")
western_frames = speeches_only[speeches_only['is_western'] == 1]['frame_dominant'].value_counts()
print(western_frames.head())

print("\nNon-Western Countries:")
non_western_frames = speeches_only[speeches_only['is_western'] == 0]['frame_dominant'].value_counts()
print(non_western_frames.head())

print("\nP5 Members:")
p5_frames = speeches_only[speeches_only['is_p5'] == 1]['frame_dominant'].value_counts()
print(p5_frames.head())

## Transformer and embeddings


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

In [ ]:
print("\n" + "="*70)
print("GENERATING SPEECH EMBEDDINGS")
print("="*70)

# Filter to speeches only
speeches_only = df[df['country_spoke'] == 1].copy()
speeches_only = speeches_only.reset_index(drop=True)

print(f"Speeches to embed: {len(speeches_only)}")

# Load pre-trained sentence transformer model
print("\nLoading sentence transformer model...")
print("Using: 'all-MiniLM-L6-v2' (fast and efficient)")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✓ Model loaded")

# Prepare texts for embedding
# Use cleaned speech text
texts_to_embed = speeches_only['speech_clean'].fillna('').tolist()

# Generate embeddings
print("\nGenerating embeddings...")
print("(This may take 1-2 minutes depending on corpus size...)")

speech_embeddings = embedding_model.encode(
    texts_to_embed,
    show_progress_bar=True,
    batch_size=32
)

print(f"✓ Generated embeddings: {speech_embeddings.shape}")
print(f"  {speech_embeddings.shape[0]} speeches")
print(f"  {speech_embeddings.shape[1]} dimensions per embedding")

# Add embeddings to dataframe
speeches_only['embedding'] = list(speech_embeddings)

# Also save embeddings as separate columns (for CSV export)
embedding_df = pd.DataFrame(
    speech_embeddings,
    columns=[f'emb_{i}' for i in range(speech_embeddings.shape[1])]
)

speeches_only = pd.concat([speeches_only.reset_index(drop=True), embedding_df], axis=1)

print("\n✓ Embeddings added to dataframe")

In [ ]:
print("\n" + "="*70)
print("SEMANTIC SIMILARITY ANALYSIS")
print("="*70)

# Calculate similarity matrix
print("\nCalculating pairwise similarities...")

similarity_matrix = cosine_similarity(speech_embeddings)

print(f"✓ Similarity matrix: {similarity_matrix.shape}")

# Analyze similarity patterns
print("\n1. Overall Similarity Statistics:")
# Get upper triangle (exclude diagonal)
upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

print(f"  Mean similarity: {upper_triangle.mean():.3f}")
print(f"  Std similarity: {upper_triangle.std():.3f}")
print(f"  Min similarity: {upper_triangle.min():.3f}")
print(f"  Max similarity: {upper_triangle.max():.3f}")

# Find most similar speech pairs
print("\n2. Top 10 Most Similar Speech Pairs:")

# Get indices of top similarities (excluding diagonal)
np.fill_diagonal(similarity_matrix, -1)  # Exclude self-similarity

top_pairs = []
for i in range(len(similarity_matrix)):
    for j in range(i+1, len(similarity_matrix)):
        top_pairs.append((i, j, similarity_matrix[i, j]))

top_pairs_sorted = sorted(top_pairs, key=lambda x: x[2], reverse=True)

for rank, (i, j, sim) in enumerate(top_pairs_sorted[:10], 1):
    speech1 = speeches_only.iloc[i]
    speech2 = speeches_only.iloc[j]

    print(f"\n{rank}. Similarity: {sim:.3f}")
    print(f"   Speech A: {speech1['country']} , Topic: {speech1['topic']})")
    print(f"   Speech B: {speech2['country']} , Topic: {speech2['topic']})")

# Find most dissimilar speech pairs
print("\n3. Top 5 Most Dissimilar Speech Pairs:")

bottom_pairs_sorted = sorted(top_pairs, key=lambda x: x[2])

for rank, (i, j, sim) in enumerate(bottom_pairs_sorted[:5], 1):
    speech1 = speeches_only.iloc[i]
    speech2 = speeches_only.iloc[j]

    print(f"\n{rank}. Similarity: {sim:.3f}")
    print(f"   Speech A: {speech1['country']} , Topic: {speech1['topic']})")
    print(f"   Speech B: {speech2['country']} , Topic: {speech2['topic']})")

In [ ]:
print("\n" + "="*70)
print("SIMILARITY MATRIX VISUALIZATION")
print("="*70)

# Create a more readable similarity matrix
# Sort by topic for better visualization
speeches_sorted = speeches_only.sort_values(['topic'])
sorted_indices = speeches_sorted.index.tolist()

# Reorder similarity matrix
similarity_sorted = similarity_matrix[sorted_indices][:, sorted_indices]

# Create labels
labels = [f"{row['country'][:8]}" for _, row in speeches_sorted.iterrows()]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Full similarity matrix
ax1 = axes[0]
im1 = ax1.imshow(similarity_sorted, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax1.set_title('Speech Similarity Matrix\n(sorted by topic)', fontsize=14, fontweight='bold') # Updated title
ax1.set_xlabel('Speech Index')
ax1.set_ylabel('Speech Index')
plt.colorbar(im1, ax=ax1, label='Cosine Similarity')


# Histogram of similarities
ax2 = axes[1]
ax2.hist(upper_triangle, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax2.axvline(upper_triangle.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {upper_triangle.mean():.3f}')
ax2.set_xlabel('Cosine Similarity')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Speech Similarities', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("DIMENSIONALITY REDUCTION (t-SNE)")
print("="*70)

print("\nReducing embeddings from 384D to 2D using t-SNE...")
print("(This may take 1-2 minutes...)")

# Apply t-SNE
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=min(30, len(speeches_only)-1),
    n_iter=1000,
    verbose=1
)

embeddings_2d = tsne.fit_transform(speech_embeddings)

print("✓ t-SNE complete")

# Add to dataframe
speeches_only['tsne_x'] = embeddings_2d[:, 0]
speeches_only['tsne_y'] = embeddings_2d[:, 1]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Color by topic
ax1 = axes[0]
topics_unique = speeches_only['topic'].unique()
topic_colors = plt.cm.Set3(np.linspace(0, 1, len(topics_unique)))

for i, topic in enumerate(topics_unique):
    mask = speeches_only['topic'] == topic
    ax1.scatter(
        speeches_only[mask]['tsne_x'],
        speeches_only[mask]['tsne_y'],
        c=[topic_colors[i]],
        label=topic,
        alpha=0.6,
        s=100
    )
ax1.set_xlabel('t-SNE Dimension 1')
ax1.set_ylabel('t-SNE Dimension 2')
ax1.set_title('Speech Embeddings by Meeting Topic', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Color by country group
ax2 = axes[1]
for group_name, condition, color in [
    ('Western', speeches_only['is_western'] == 1, 'blue'),
    ('Non-Western', speeches_only['is_western'] == 0, 'red'),
]:
    mask = condition
    ax2.scatter(
        speeches_only[mask]['tsne_x'],
        speeches_only[mask]['tsne_y'],
        c=color,
        label=group_name,
        alpha=0.6,
        s=100
    )
ax2.set_xlabel('t-SNE Dimension 1')
ax2.set_ylabel('t-SNE Dimension 2')
ax2.set_title('Speech Embeddings by Country Group', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("CLUSTERING SPEECHES BASED ON EMBEDDINGS")
print("="*70)

# Try different numbers of clusters
print("\nTesting different cluster numbers...")

from sklearn.metrics import silhouette_score

cluster_range = range(2, 8)
silhouette_scores = []

for n_clusters in cluster_range:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(speech_embeddings)
    score = silhouette_score(speech_embeddings, cluster_labels)
    silhouette_scores.append(score)
    print(f"  {n_clusters} clusters: silhouette score = {score:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(cluster_range, silhouette_scores, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Use optimal number of clusters
optimal_k = cluster_range[np.argmax(silhouette_scores)]
print(f"\n✓ Optimal number of clusters: {optimal_k}")

# Perform final clustering
print(f"\nClustering with k={optimal_k}...")
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
speeches_only['cluster'] = kmeans_final.fit_predict(speech_embeddings)

print("✓ Clustering complete")

# Analyze clusters
print("\n" + "="*70)
print("CLUSTER ANALYSIS")
print("="*70)

for cluster_id in range(optimal_k):
    cluster_data = speeches_only[speeches_only['cluster'] == cluster_id]

    print(f"\n{'─'*70}")
    print(f"Cluster {cluster_id} (n={len(cluster_data)})")
    print(f"{'─'*70}")

    # Topic distribution
    print(f"\nTopic distribution:")
    topic_dist = cluster_data['topic'].value_counts()
    for topic, count in topic_dist.items():
        print(f"  {topic}: {count}")

    # Top countries
    print(f"\nTop countries:")
    country_dist = cluster_data['country'].value_counts().head(3)
    for country, count in country_dist.items():
        print(f"  {country}: {count}")

    # Dominant frame
    if 'frame_dominant' in cluster_data.columns:
        print(f"\nDominant frames:")
        frame_dist = cluster_data['frame_dominant'].value_counts().head(2)
        for frame, count in frame_dist.items():
            print(f"  {frame}: {count}")

    # Example speech
    example = cluster_data.iloc[0]
    print(f"\nExample speech:")
    print(f"  {example['country']} (Topic: {example['topic']})")
    print(f"  {example['speech_clean'][:200]}...")

In [ ]:
print("\n" + "="*70)
print("CLUSTER VISUALIZATION")
print("="*70)

# Plot clusters on t-SNE
fig, ax = plt.subplots(figsize=(8, 7))

# Plot 1: Clusters
scatter = ax.scatter(
    speeches_only['tsne_x'],
    speeches_only['tsne_y'],
    c=speeches_only['cluster'],
    cmap='Set3',
    alpha=0.6,
    s=100
)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title(f'Speech Clusters (k={optimal_k})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add cluster labels
for cluster_id in range(optimal_k):
    cluster_center = speeches_only[speeches_only['cluster'] == cluster_id][['tsne_x', 'tsne_y']].mean()
    ax.text(cluster_center['tsne_x'], cluster_center['tsne_y'],
            f'C{cluster_id}', fontsize=16, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("ZERO-SHOT CLASSIFICATION")
print("="*70)

print("\nLoading zero-shot classification model...")
print("(This may take a minute...)")

# Load zero-shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

print("✓ Model loaded")

# Define candidate labels for classification
candidate_labels = [
    "strongly supports the resolution",
    "moderately supports with reservations",
    "neutral stance calling for dialogue",
    "opposes the resolution approach",
    "strongly opposes and rejects resolution"
]

print(f"\nCandidate labels: {len(candidate_labels)}")
for label in candidate_labels:
    print(f"  • {label}")

# Classify a sample of speeches
print("\nClassifying speeches...")
print("(Testing on first 10 speeches for speed...")

sample_speeches = speeches_only.head(10)
zero_shot_results = []

for idx, row in sample_speeches.iterrows():
    text = row['speech_clean'][:512]  # Truncate to 512 chars for speed

    result = classifier(text, candidate_labels, multi_label=False)

    zero_shot_results.append({
        'country': row['country'],
        'predicted_stance': result['labels'][0],
        'confidence': result['scores'][0],
        'top_3_labels': result['labels'][:3],
        'top_3_scores': result['scores'][:3]
    })

    print(f"  {idx+1}. {row['country']}: {result['labels'][0]} ({result['scores'][0]:.2f})")

# Create results dataframe
zero_shot_df = pd.DataFrame(zero_shot_results)

print("\n" + "="*70)
print("ZERO-SHOT CLASSIFICATION RESULTS")
print("="*70)

print("\nPredicted Stance Distribution:")
print(zero_shot_df['predicted_stance'].value_counts())

print("\nExample Predictions:")
print(zero_shot_df[['country', 'predicted_stance', 'confidence']].to_string(index=False))